# Healthcare RAG System Lab
## Overview

In this lab, you'll take on the role of a junior data scientist at a healthcare technology company that specializes in creating educational resources for patients. Your team has been tasked with developing a system that can automatically generate informative responses to common patient questions about medical conditions, treatments, and wellness practices.

The challenge is to ensure these responses are both accurate and grounded in authoritative medical information. Your specific assignment is to implement a Retrieval-Augmented Generation (RAG) system that can:
1. Understand patient questions about various health topics
2. Retrieve relevant information from a trusted knowledge base
3. Generate helpful, accurate responses based on that information
4. Avoid "hallucinated" content that could potentially misinform patients

This lab follows the generative AI implementation process we've studied, with particular focus on:
- Data Strategy and Knowledge Foundation
- Model Selection and Generation Control
- Evaluation Framework Development

## Setup

First, let's import the necessary libraries:

In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

/usr/local/Caskroom/miniconda/base/envs/learn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-20 05:20:15.420621: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cpu


## Part 1: Knowledge Base Setup

Let's create a sample medical knowledge base with information about common health conditions, treatments, and wellness practices:

In [2]:
# Create a sample medical knowledge base
knowledge_base = pd.DataFrame({
    'content': [
        "Diabetes is a chronic condition that affects how your body turns food into energy. There are three main types: Type 1, Type 2, and gestational diabetes. Type 2 diabetes is the most common form, accounting for about 90-95% of diabetes cases.",
        "Type 1 diabetes is an autoimmune reaction that stops your body from making insulin. Symptoms include increased thirst, frequent urination, hunger, fatigue, and blurred vision. It's usually diagnosed in children, teens, and young adults.",
        "Type 2 diabetes occurs when your body becomes resistant to insulin or doesn't make enough insulin. Risk factors include being overweight, being 45 years or older, having a parent or sibling with type 2 diabetes, and being physically active less than 3 times a week.",
        "Managing diabetes involves monitoring blood sugar levels, taking medications as prescribed, eating a healthy diet, maintaining a healthy weight, and getting regular physical activity. It's important to work with healthcare providers to develop a management plan.",
        "Hypertension, or high blood pressure, is when the force of blood pushing against the walls of your arteries is consistently too high. It's often called the 'silent killer' because it typically has no symptoms but significantly increases the risk of heart disease and stroke.",
        "Blood pressure is measured using two numbers: systolic (top number) and diastolic (bottom number). Normal blood pressure is less than 120/80 mm Hg. Hypertension is diagnosed when readings are consistently 130/80 mm Hg or higher.",
        "Lifestyle changes to manage hypertension include reducing sodium in your diet, getting regular physical activity, maintaining a healthy weight, limiting alcohol, quitting smoking, and managing stress. Medications may also be prescribed if lifestyle changes aren't enough.",
        "Regular physical activity offers numerous health benefits, including weight management, reduced risk of heart disease, strengthened bones and muscles, improved mental health, and enhanced ability to perform daily activities. Adults should aim for at least 150 minutes of moderate-intensity activity per week.",
        "A balanced diet should include a variety of fruits, vegetables, whole grains, lean proteins, and healthy fats. It's recommended to limit intake of added sugars, sodium, saturated fats, and processed foods. Proper nutrition helps prevent chronic diseases and supports overall health.",
        "Vaccination is one of the most effective ways to prevent infectious diseases. Vaccines work by helping the body recognize and fight specific pathogens. Common adult vaccines include influenza (flu), Tdap (tetanus, diphtheria, pertussis), shingles, and pneumococcal vaccines."
    ],
    'metadata': [
        {'topic': 'diabetes', 'subtopic': 'overview', 'source': 'medical_guidelines', 'last_updated': '2023-06-10'},
        {'topic': 'diabetes', 'subtopic': 'type1', 'source': 'medical_guidelines', 'last_updated': '2023-06-10'},
        {'topic': 'diabetes', 'subtopic': 'type2', 'source': 'medical_guidelines', 'last_updated': '2023-06-10'},
        {'topic': 'diabetes', 'subtopic': 'management', 'source': 'medical_guidelines', 'last_updated': '2023-06-10'},
        {'topic': 'hypertension', 'subtopic': 'overview', 'source': 'medical_guidelines', 'last_updated': '2023-07-22'},
        {'topic': 'hypertension', 'subtopic': 'diagnosis', 'source': 'medical_guidelines', 'last_updated': '2023-07-22'},
        {'topic': 'hypertension', 'subtopic': 'management', 'source': 'medical_guidelines', 'last_updated': '2023-07-22'},
        {'topic': 'wellness', 'subtopic': 'physical_activity', 'source': 'health_promotion', 'last_updated': '2023-05-15'},
        {'topic': 'wellness', 'subtopic': 'nutrition', 'source': 'health_promotion', 'last_updated': '2023-05-15'},
        {'topic': 'prevention', 'subtopic': 'vaccination', 'source': 'medical_guidelines', 'last_updated': '2023-08-05'}
    ]
})

print(f"Knowledge base loaded with {len(knowledge_base)} entries")
knowledge_base.head(2)

Knowledge base loaded with 10 entries


,content,metadata
0,Diabetes is a chronic condition that affects h...,"{'topic': 'diabetes', 'subtopic': 'overview', ..."
1,Type 1 diabetes is an autoimmune reaction that...,"{'topic': 'diabetes', 'subtopic': 'type1', 'so..."


### Task 1: Create Document Embeddings

Complete the function below to create embeddings for each document in the knowledge base. These embeddings will be used to find relevant documents based on patient queries.

In [3]:
def create_document_embeddings(documents):
    """
    Create embeddings for a list of documents.
    
    Args:
        documents: List of text documents to embed
        
    Returns:
        Numpy array of document embeddings
    """
    # Initialize a sentence transformer model
    # Recommended: 'sentence-transformers/all-mpnet-base-v2' or similar
    embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2') # None  # Replace with your code
    
    # Generate embeddings for all documents
    # Hint: Use the model.encode() method
    document_embeddings = embedding_model.encode(knowledge_base['content'].tolist(), 
                                            show_progress_bar=True)# None  # Replace with your code
    
    return document_embeddings

# Extract document content
documents = knowledge_base['content'].tolist()

# Create document embeddings
document_embeddings = create_document_embeddings(documents)

# Verify the shape of embeddings
if document_embeddings is not None:
    print(f"Generated embeddings with shape: {document_embeddings.shape}")
else:
    print("Embeddings not created yet.")

Batches:   0%|                                           | 0/1 [00:00<?, ?it/s]/usr/local/Caskroom/miniconda/base/envs/learn-env/lib/python3.12/site-packages/transformers/models/mpnet/modeling_mpnet.py:954: UserWarning: cumsum_out_mps supported by MPS on MacOS 13+, please upgrade (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/UnaryOps.mm:425.)
  incremental_indices = torch.cumsum(mask, dim=1).type_as(mask) * mask
Batches: 100%|███████████████████████████████████| 1/1 [00:01<00:00,  1.76s/it]

Generated embeddings with shape: (10, 768)


## Part 2: Implementing the Retrieval Component

Now, let's implement the function to retrieve relevant documents based on a patient query.

In [7]:
def retrieve_documents(query, embeddings, contents, metadata, top_k=3, threshold=0.3):
    """
    Retrieve the most relevant documents for a given query.
    
    Args:
        query: The patient's question
        embeddings: The precomputed document embeddings
        contents: The text content of the documents
        metadata: The metadata for each document
        top_k: Maximum number of documents to retrieve
        threshold: Minimum similarity score to include a document
        
    Returns:
        List of (content, metadata, similarity_score) tuples
    """
    # Get or initialize the embedding model (same as in create_document_embeddings)
    embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2') 
    
    # Embed the query
    query_embedding = embedding_model.encode( [query] ) #None  # Replace with your code
    print(query_embedding.shape)
    
    # Calculate similarity scores between query and all documents
    # Hint: Use cosine_similarity
    similarities = cosine_similarity(query_embedding, embeddings)[0] #None  # Replace with your code
    
    # Filter by threshold and get top k results
    # Hint: Use list comprehension, sorting, and slicing
    filtered_indices = [ i for i, score in enumerate(similarities) if score >= threshold ]
    top_indices = sorted(filtered_indices, key=lambda i: similarities[i], reverse=True)[:top_k]
    
    # Return the top documents with their metadata and scores
    results = [(contents[i],metadata[i] ,similarities[i]) for i in top_indices ]  # Replace with your code
    
    return results

# Test the retrieval function with a sample query
if document_embeddings is not None:
    sample_query = "What are the symptoms of Type 1 diabetes?"
    retrieved_docs = retrieve_documents(
        query=sample_query,
        embeddings=document_embeddings,
        contents=documents,
        metadata=knowledge_base['metadata'].tolist(),
        top_k=2
    )
    
    print(f"Query: {sample_query}")
    print("\nRetrieved Documents:")
    for i, (content, meta, score) in enumerate(retrieved_docs):
        print(f"{i+1}. [{score:.4f}] {content[:100]}...")
        print(f"   Topic: {meta['topic']}, Subtopic: {meta['subtopic']}")
else:
    print("Cannot test retrieval without document embeddings.")

(1, 768)
Query: What are the symptoms of Type 1 diabetes?

Retrieved Documents:
1. [0.7585] Type 1 diabetes is an autoimmune reaction that stops your body from making insulin. Symptoms include...
   Topic: diabetes, Subtopic: type1
2. [0.4625] Diabetes is a chronic condition that affects how your body turns food into energy. There are three m...
   Topic: diabetes, Subtopic: overview


## Part 3: Building the Generation Component

Now, let's implement the generation component that will use the retrieved documents to create informative responses.

In [8]:
# Initialize the generative model
def initialize_generator(model_name="gpt2"):
    """
    Initialize the generative model and tokenizer.
    
    Args:
        model_name: Name of the pretrained model to use
        
    Returns:
        Tokenizer and model objects
    """
    # Load the tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name) # None  # Replace with your code
    model = AutoModelForCausalLM.from_pretrained(model_name) #None  # Replace with your code
    
    # Set padding token if needed
    # Check if pad_token exists, if not set it to eos_token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer, model

# Initialize the generator
tokenizer, model = initialize_generator()
if tokenizer and model:
    print(f"Initialized {model.config._name_or_path} with {model.num_parameters()} parameters")

Initialized gpt2 with 124439808 parameters


In [10]:
def generate_rag_response(query, contents, metadata, document_embeddings, tokenizer, model, max_length=100):
    """
    Generate a response using Retrieval-Augmented Generation.
    
    Args:
        query: The patient's question
        contents: List of document contents
        metadata: List of document metadata
        document_embeddings: Precomputed embeddings for the documents
        tokenizer: The tokenizer for the language model
        model: The language model for generation
        max_length: Maximum response length
        
    Returns:
        Dictionary with the generated response and the retrieved documents
    """
    # Retrieve relevant documents for the query
    retrieved_docs = retrieve_documents(
        query, 
        document_embeddings, 
        contents,
        metadata
    ) #None  
    
    # Format prompt with retrieved context
    # Hint: If no relevant documents are found, generate without context
    # Otherwise, include retrieved documents as context
    if not retrieved_docs:
        # If no relevant documents found, generate without context
        prompt = f"Question: {query}\nAnswer:"
    else:
        # Format prompt with retrieved context
        context = "\n".join([f"- {doc[0]}" for doc in retrieved_docs])
        prompt = f"Context information:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True) #None  # Replace with your code
    
    # Generate the response
    # Consider including temperature, top_k, and do_sample parameters (will otherwise use greedy method)
    with torch.no_grad():
        output_sequences = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=len(inputs["input_ids"][0]) + max_length,
            temperature=0.7, 
            top_p=0.9, 
            do_sample=True, 
            pad_token_id=tokenizer.pad_token_id
        ) # None  # Replace with your code
    
    # Decode the response and extract the generated text
    response = tokenizer.decode( output_sequences[0], skip_special_tokens=True ) #None  # Replace with your code
    response = response.replace(prompt, "").strip()
    
    # Return the results
    return {
        "query": query,
        "response": response,
        "retrieved_documents": retrieved_docs
    }

# Test the RAG system with several queries
if document_embeddings is not None and tokenizer and model:
    test_queries = [
        "What are the different types of diabetes?",
        "How can I manage my high blood pressure through lifestyle changes?",
        "Why is regular physical activity important for health?",
        "What vaccines should adults consider getting?"
    ]
    
    for query in test_queries:
        print(f"\nQuery: {query}")
        result = generate_rag_response(
            query=query,
            contents=documents,
            metadata=knowledge_base['metadata'].tolist(),
            document_embeddings=document_embeddings,
            tokenizer=tokenizer,
            model=model
        )
        
        print("\nRetrieved Documents:")
        for i, (doc, meta, score) in enumerate(result["retrieved_documents"]):
            print(f"{i+1}. [{score:.4f}] Topic: {meta['topic']}, Subtopic: {meta['subtopic']}")
        
        print(f"\nGenerated Response:\n{result['response']}")
        print("-" * 80)
else:
    print("Cannot test generation without embeddings or model.")


Query: What are the different types of diabetes?
(1, 768)

Retrieved Documents:
1. [0.7130] Topic: diabetes, Subtopic: overview
2. [0.6430] Topic: diabetes, Subtopic: type1
3. [0.6042] Topic: diabetes, Subtopic: type2

Generated Response:
Diabetes is a chronic condition that affects how your body turns food into energy. There are three main types: Type 1, Type 2, and gestational diabetes. Type 2 diabetes is the most common form, accounting for about 90-95% of diabetes cases. - Type 1 diabetes is an autoimmune reaction that stops your body from making insulin. Symptoms include increased thirst, frequent urination, hunger, fatigue, and blurred vision. It's usually diagnosed in children, teens, and young adults. - Type
--------------------------------------------------------------------------------

Query: How can I manage my high blood pressure through lifestyle changes?
(1, 768)

Retrieved Documents:
1. [0.7775] Topic: hypertension, Subtopic: management
2. [0.4690] Topic: hypertension,

## Part 4: Evaluation and Analysis

Let's implement a basic evaluation function to assess the quality of our generated responses.

In [11]:
def evaluate_response(response_data):
    """
    Evaluate the quality of a generated response based on various criteria.
    
    Args:
        response_data: Dictionary containing the query, response, and retrieved docs
        
    Returns:
        Evaluation metrics
    """
    # Implement at least two evaluation metrics
    # Suggestions:
    # 1. Content relevance - Check if response mentions terms from retrieved docs
    # 2. Response length appropriateness - Check if length is suitable for query
    # 3. Medical terminology usage - Check if appropriate medical terms are included
    medical_terms = [
        "diabetes", "insulin", "glucose", "hypertension", "blood pressure",
        "systolic", "diastolic", "cardiovascular", "cholesterol", "nutrition",
        "obesity", "physical activity", "vaccination", "immune", "prevention"
    ]
    
    metrics = {}  # Replace with your code

    # Metric 1: Medical terminology usage
    response_text = response_data["response"].lower()
    terms_found = [term for term in medical_terms if term in response_text]
    metrics["medical_terms_count"] = len( terms_found )

    # Metric 2: Response length appropriateness
    response_length = len(response_text) # number of characters
    metrics["response_length"] = response_length
    if response_length < 50:
        metrics["length_category"] = "too_short"
    elif response_length > 500:
        metrics["length_category"] = "too_long"
    else:
        metrics["length_category"] = "reasonable"

    # Metric 3: Content overlap with retrieved documents
    retrieved_text = " ".join([doc[0].lower() for doc in response_data["retrieved_documents"]])
    # simple word overlap example
    response_words = set(response_text.split())
    retrieved_words = set(retrieved_text.split())
    overlap = response_words.intersection(retrieved_words)
    metrics["content_overlap_count"] = len(overlap)
    
    
    return metrics

# Evaluate the responses for our test queries
if 'test_queries' in locals() and document_embeddings is not None and tokenizer and model:
    for query in test_queries:
        result = generate_rag_response(
            query=query,
            contents=documents,
            metadata=knowledge_base['metadata'].tolist(),
            document_embeddings=document_embeddings,
            tokenizer=tokenizer,
            model=model
        )
        
        metrics = evaluate_response(result)
        print(f"Query: {query}")
        print(f"Evaluation Metrics: {metrics}")
        print("-" * 80)
else:
    print("Cannot evaluate without test queries or necessary components.")

(1, 768)
Query: What are the different types of diabetes?
Evaluation Metrics: {'medical_terms_count': 3, 'response_length': 486, 'length_category': 'reasonable', 'content_overlap_count': 62}
--------------------------------------------------------------------------------
(1, 768)
Query: How can I manage my high blood pressure through lifestyle changes?
Evaluation Metrics: {'medical_terms_count': 1, 'response_length': 421, 'length_category': 'reasonable', 'content_overlap_count': 3}
--------------------------------------------------------------------------------
(1, 768)
Query: Why is regular physical activity important for health?
Evaluation Metrics: {'medical_terms_count': 1, 'response_length': 523, 'length_category': 'too_long', 'content_overlap_count': 12}
--------------------------------------------------------------------------------
(1, 768)
Query: What vaccines should adults consider getting?
Evaluation Metrics: {'medical_terms_count': 1, 'response_length': 411, 'length_category

## Reflection Questions

Answer the following questions about your RAG implementation and its potential applications in healthcare:

### How does the RAG approach improve factual accuracy compared to regular generation?

<i>
In a regular generation setup without retrieval, the model can only draw on the knowledge it learned during training. That knowledge may be incomplete, outdated, or incorrect, which increases the risk of hallucinations.
In the RAG approach, the system first retrieves the most relevant documents from the knowledge base and inserts their text directly into the prompt as context. The model then generates its answer conditioned on that retrieved information.
Because the relevant facts are placed right in front of the model, it is much more likely to reuse the provided information instead of inventing new medical claims.

    
For example, in the diabetes query the generated response began with “Diabetes is a chronic condition that affects how your body turns food into energy. There are three main types…”, which is nearly a direct copy of the first document in the knowledge base. This shows the retrieved context shaping the output.
</i>

### What are potential challenges or limitations of your current implementation?


<i>

Even when the retrieval step correctly identified relevant documents, the generated responses had several limitations. GPT-2 often repeated or lightly rephrased the retrieved text and then drifted, invented extra questions or unrelated advice, or produced answers that were only partially grounded in the context.

These issues are partly due to the model’s limited capacity. GPT-2 struggles to stay on topic, synthesize information, or carefully follow instructions, whereas a larger model would generally produce more coherent and relevant answers from the same retrieved context.

Additional challenges exist in the retrieval step itself. The fixed similarity threshold and top-k value can pull in weakly related documents or miss useful ones, the embedding model may not capture medical nuance perfectly, and there is no re-ranking or further filtering beyond cosine similarity.

The simple prompt format (“Context: … Question: … Answer:”) also provides little guidance, such as instructions to use only the provided context, be concise, or cite sources. A longer, more structured prompt would improve robustness. Finally, the evaluation metrics are basic—they measure term counts, length, and word overlap but do not assess factual correctness, coherence, or whether the response actually answers the question.

</i>



### How might you enhance this system for a production healthcare environment?

<i> 

For a production healthcare environment, several enhancements would be important.


The knowledge base should be expanded significantly in size and quality. Sources should come from authoritative medical literature and guidelines (for example, peer-reviewed papers, CDC, NIH, or professional society recommendations) rather than a small static sample. A scheduled process (such as a daily or weekly job) could keep the knowledge base updated with new or revised information, while the existing document-plus-metadata structure could be retained and enriched with additional fields (e.g., evidence level, publication date, or audience).


Retrieval reliability could be improved by using higher-quality domain-specific embeddings, adding a re-ranking step after the initial cosine similarity search, supporting hybrid (keyword + semantic) search, and applying metadata filters (for example, by topic, recency, or source reliability).


On the generation side, switching to a larger, more capable model would improve coherence and instruction-following. Stronger, more structured prompting (with explicit instructions to stay within the retrieved context, be concise, and avoid speculation) plus output guardrails would help keep responses on track. Automatically including citations or source references from the retrieved documents would also increase transparency.


Evaluation and monitoring should move beyond simple term counts and length checks to include factual consistency measures, human-rated quality samples, and ongoing logging of retrieval scores and user feedback.


Finally, extra safeguards are essential for medical content. Responses could carry clear disclaimers that the information is educational and not a substitute for professional medical advice. A confidence or relevance threshold could trigger human review (human-in-the-loop) for low-confidence or high-risk queries, and the system should refuse or escalate questions that fall outside its knowledge base or require diagnosis or treatment decisions.

</i>

### What ethical considerations are particularly important for healthcare content generation?


<i>

Accuracy and the avoidance of misinformation are especially critical in healthcare because incorrect information can directly affect patients’ decisions and health outcomes. Generated content must therefore be carefully grounded and verified.
If the system presents generated responses as medical advice, there is a risk that users may rely on them for diagnosis, treatment, or medication decisions instead of consulting a qualified professional. This could lead to delayed care, inappropriate self-treatment, or other harm.
When the system encounters uncertainty, incomplete information, or questions that require clinical judgment, it should escalate to a human-in-the-loop process (for example, by surfacing a low-confidence warning and recommending consultation with a licensed healthcare provider) rather than attempting to answer.
The organization deploying the system has clear responsibilities around transparency, disclaimers, and data privacy. Responses should include prominent disclaimers that the content is for educational purposes only and is not a substitute for professional medical advice. All applicable health-privacy regulations (such as HIPAA in the United States) must be followed so that patient data remains protected; failure to do so can create legal liability.
Equity and accessibility are also important. The system should be designed to support varying levels of health literacy, multiple languages where possible, and careful attention to bias in both the knowledge base and the model so that it does not disadvantage particular patient populations.

</i>

